# 05 Create SFINCS Scenarios

Create the full wave-coupled SFINCS scenario folders for Slurm. This notebook does not submit jobs; it verifies inputs, builds configured `data/sfincs/scenarios/evt_*` folders, and writes a compact launch plan for the DSAI runner. Each event folder carries per-event SnapWave and precipitation forcing plus boundary water-level files; large static base-model files are hardlinked where the filesystem allows it.


> **Stage Contract**
>
> Requires: wave-coupled base model, event catalog, rainfall, waves, surge, soil moisture
>
> Produces: data/sfincs/scenarios/evt_* scenario folders
>
> Next: DSAI sync and Slurm truth-set runs, then 06_evaluate.ipynb

In [1]:
# Load local packages and this Location Workspace.
import sys
from pathlib import Path

repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
src_root = repo_root / "src"
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))
workspace = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.yaml").exists())
config_yaml = workspace / "config.yaml"

import json
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import display

from sfincs_runs.config import load_runtime
from sfincs_runs.scenarios.create_events import parse_args, build_scenarios

config, paths = load_runtime(config_yaml)
scenario_limit = 500
force_rebuild = True
resume_existing = False
workers_per_cpu_node = 1
array_chunks = 16
max_concurrent_cpu_nodes = 4

display(pd.Series({
    "location": paths["location_name"],
    "base_model_root": str((config.get("coastal_wave_coupling", {}).get("quadtree", {}).get("base_model_root") or paths["base_model_root"])),
    "scenarios_root": str(paths["scenarios_root"]),
    "storage_root": str(paths["storage_root"]),
    "run_root": str(paths["run_root"]),
    "scenario_limit": scenario_limit,
    "force_rebuild": force_rebuild,
    "resume_existing": resume_existing,
}, name="create_scenarios_paths"))


location                                                  marshfield
base_model_root                   data/sfincs/base_quadtree_snapwave
scenarios_root     /home/grahamhults/projects/Flood-RM/locations/...
storage_root       /home/grahamhults/projects/Flood-RM/locations/...
run_root           /home/grahamhults/projects/Flood-RM/locations/...
scenario_limit                                                   500
force_rebuild                                                   True
resume_existing                                                False
Name: create_scenarios_paths, dtype: object

## Required Inputs

The scenario builder consumes the wave-coupled base model and Event Catalog outputs. Missing inputs here mean the cluster job is not ready to submit.


In [2]:
wave_base = (
    workspace / config.get("coastal_wave_coupling", {}).get("quadtree", {}).get("base_model_root", "data/sfincs/base_quadtree_snapwave")
)
required = {
    "wave_base_model": wave_base,
    "base_sfincs_inp": wave_base / "sfincs.inp",
    "base_boundary": wave_base / "sfincs.bnd",
    "base_snapwave_boundary": wave_base / "snapwave.bnd",
    "base_runup_gauges": wave_base / "sfincs.rug",
    "event_catalog": paths["design_outputs_root"] / "catalog" / "event_catalog.csv",
    "event_catalog_audit": paths["design_outputs_root"] / "catalog" / "event_catalog_audit.json",
    "surge_members": paths["design_outputs_root"] / "events" / "surge_event_members.nc",
    "data_catalog": paths["data_catalog"],
}
status = pd.DataFrame([
    {"artifact": key, "path": str(path), "exists": Path(path).exists()}
    for key, path in required.items()
])
display(status)
missing = status.loc[~status["exists"], "artifact"].tolist()
if missing:
    raise FileNotFoundError("Missing required scenario-build inputs: " + ", ".join(missing))


,artifact,path,exists
0,wave_base_model,/home/grahamhults/projects/Flood-RM/locations/...,True
1,base_sfincs_inp,/home/grahamhults/projects/Flood-RM/locations/...,True
2,base_boundary,/home/grahamhults/projects/Flood-RM/locations/...,True
3,base_snapwave_boundary,/home/grahamhults/projects/Flood-RM/locations/...,True
4,base_runup_gauges,/home/grahamhults/projects/Flood-RM/locations/...,True
5,event_catalog,/home/grahamhults/projects/Flood-RM/locations/...,True
6,event_catalog_audit,/home/grahamhults/projects/Flood-RM/locations/...,True
7,surge_members,/home/grahamhults/projects/Flood-RM/locations/...,True
8,data_catalog,/home/grahamhults/projects/Flood-RM/locations/...,True


## Build Full Wave-Coupled Scenario Folders

This is the production staging step for the cluster. It writes one `evt_*` folder per selected Event Catalog row with coastal water level, SnapWave time series, AORC precipitation forcing, soil initial wetness, and a `forcing_manifest.json`. Static quadtree files are hardlinked first and copied only if hardlinks are unavailable.


In [3]:
builder_argv = [
    "--config", str(config_yaml),
    "--limit", str(scenario_limit),
    "--include-waves",
    "--include-precip",
    "--zsini-mode", "boundary_t0",
]
if force_rebuild:
    builder_argv.append("--force")
elif resume_existing:
    builder_argv.append("--resume")
else:
    raise RuntimeError("Choose force_rebuild=True or resume_existing=True before building scenarios.")

print("python -m sfincs_runs build_scenarios " + " ".join(builder_argv))
args = parse_args(builder_argv)
report = build_scenarios(args)
display(report[[
    "event_id", "run_root", "run_duration_hours", "forcing_variable",
    "expected_has_waves", "expected_has_precip", "netamprfile",
    "rainfall_window_alignment", "expected_zsini_m",
]].head())


python -m sfincs_runs build_scenarios --config /home/grahamhults/projects/Flood-RM/locations/marshfield/config.yaml --limit 500 --include-waves --include-precip --zsini-mode boundary_t0 --force
2026-05-28 08:37:38,682 - hydromt.model.model - model - INFO - Initializing sfincs model from hydromt_sfincs (v2.0.0rc1).
2026-05-28 08:37:38,683 - hydromt.model.model - model - WARNING - No region component found in components.
2026-05-28 08:37:42,418 - hydromt.data_catalog.sources.data_source - data_source - INFO - Reading event_precip RasterDataset data from /home/grahamhults/projects/Flood-RM/locations/marshfield/data/sfincs/scenarios/evt_0001/aorc_precip_for_sfincs.nc
2026-05-28 08:37:43,057 - hydromt.model.model - model - INFO - Initializing sfincs model from hydromt_sfincs (v2.0.0rc1).
2026-05-28 08:37:43,058 - hydromt.model.model - model - WARNING - No region component found in components.
2026-05-28 08:37:46,514 - hydromt.data_catalog.sources.data_source - data_source - INFO - Reading e

,event_id,run_root,run_duration_hours,forcing_variable,expected_has_waves,expected_has_precip,netamprfile,rainfall_window_alignment,expected_zsini_m
0,evt_0001,/home/grahamhults/projects/Flood-RM/locations/...,144.0,water_level_total,True,True,sfincs_netampr.nc,wettest,0.964087
1,evt_0002,/home/grahamhults/projects/Flood-RM/locations/...,144.0,water_level_total,True,True,sfincs_netampr.nc,wettest,-0.020447
2,evt_0003,/home/grahamhults/projects/Flood-RM/locations/...,144.0,water_level_total,True,True,sfincs_netampr.nc,wettest,1.060264
3,evt_0004,/home/grahamhults/projects/Flood-RM/locations/...,144.0,water_level_total,True,True,sfincs_netampr.nc,wettest,0.947679
4,evt_0005,/home/grahamhults/projects/Flood-RM/locations/...,144.0,water_level_total,True,True,sfincs_netampr.nc,wettest,1.152052


## Scenario Folder Audit

A quick filesystem and NetCDF audit before Slurm submission. This catches missing forcing files and the stale precipitation failure mode where `sfincs_netampr.nc` exists but keeps an old event clock or contains only NaNs.


In [4]:
scenario_root = paths["scenarios_root"]
event_dirs = sorted(p for p in scenario_root.glob("evt_*") if p.is_dir())
required_event_files = [
    "sfincs.inp", "sfincs.bnd", "sfincs.bzs", "forcing_manifest.json",
    "snapwave.bnd", "snapwave.bhs", "snapwave.btp", "snapwave.bwd", "snapwave.bds",
    "aorc_precip_for_sfincs.nc", "sfincs_netampr.nc", "sfincs.smax", "sfincs.seff", "sfincs.ks",
]

def netampr_status(event_dir, manifest):
    netampr = event_dir / str(manifest.get("netamprfile") or "sfincs_netampr.nc")
    if not netampr.exists():
        return "missing"
    run_start = manifest.get("run_start")
    run_stop = manifest.get("run_stop")
    if not run_start or not run_stop:
        return "missing_run_window"
    try:
        with xr.open_dataset(netampr) as ds:
            if "time" not in ds:
                return "missing_time_coord"
            times = pd.to_datetime(ds["time"].values)
            if len(times) == 0:
                return "empty_time_coord"
            if times[0] != pd.Timestamp(run_start) or times[-1] != pd.Timestamp(run_stop):
                return f"stale_time:{times[0]}..{times[-1]}"
            finite_any = False
            for name in ds.data_vars:
                values = ds[name].values
                if np.issubdtype(values.dtype, np.number) and np.isfinite(values).any():
                    finite_any = True
                    break
            return "ok" if finite_any else "all_nan"
    except Exception as exc:
        return f"error:{type(exc).__name__}"

audit_rows = []
for event_dir in event_dirs[: min(len(event_dirs), scenario_limit)]:
    missing = [name for name in required_event_files if not (event_dir / name).exists()]
    manifest = json.loads((event_dir / "forcing_manifest.json").read_text(encoding="utf-8")) if (event_dir / "forcing_manifest.json").exists() else {}
    audit_rows.append({
        "event_id": event_dir.name,
        "missing_files": ", ".join(missing),
        "netampr_status": netampr_status(event_dir, manifest),
        "run_duration_hours": manifest.get("run_duration_hours"),
        "forcing_variable": manifest.get("forcing_variable"),
        "waves": manifest.get("expected_has_waves"),
        "precip": manifest.get("expected_has_precip"),
    })
audit_frame = pd.DataFrame(audit_rows)
missing_count = int(audit_frame["missing_files"].astype(bool).sum()) if len(audit_frame) else 0
bad_netampr_count = int((audit_frame["netampr_status"] != "ok").sum()) if len(audit_frame) else 0
display(pd.Series({
    "scenario_root": str(scenario_root),
    "event_folders": len(event_dirs),
    "audited_folders": len(audit_frame),
    "folders_with_missing_files": missing_count,
    "folders_with_bad_netampr": bad_netampr_count,
}, name="scenario_folder_audit"))
display(audit_frame.head(12))
if len(event_dirs) < scenario_limit:
    raise RuntimeError(f"Expected at least {scenario_limit} scenario folders, found {len(event_dirs)}")
if missing_count:
    bad = audit_frame[audit_frame["missing_files"].astype(bool)].head(5)
    raise RuntimeError("Scenario folders missing required forcing files:\n" + bad.to_string(index=False))
if bad_netampr_count:
    bad = audit_frame[audit_frame["netampr_status"] != "ok"].head(5)
    raise RuntimeError("Scenario folders have invalid precipitation NetCDFs:\n" + bad.to_string(index=False))


scenario_root                 /home/grahamhults/projects/Flood-RM/locations/...
event_folders                                                               500
audited_folders                                                             500
folders_with_missing_files                                                    0
folders_with_bad_netampr                                                      0
Name: scenario_folder_audit, dtype: object

,event_id,missing_files,netampr_status,run_duration_hours,forcing_variable,waves,precip
0,evt_0001,,ok,144.0,water_level_total,True,True
1,evt_0002,,ok,144.0,water_level_total,True,True
2,evt_0003,,ok,144.0,water_level_total,True,True
3,evt_0004,,ok,144.0,water_level_total,True,True
4,evt_0005,,ok,144.0,water_level_total,True,True
5,evt_0006,,ok,144.0,water_level_total,True,True
6,evt_0007,,ok,144.0,water_level_total,True,True
7,evt_0008,,ok,144.0,water_level_total,True,True
8,evt_0009,,ok,144.0,water_level_total,True,True
9,evt_0010,,ok,144.0,water_level_total,True,True


## Cluster Launch Plan

The Slurm script runs prepared scenario folders and keeps completed outputs compact: `sfincs_map.nc`, `sfincs_his.nc`, logs, manifest, and small boundary/structure receipts. It does not retain expensive animations, restart files, precipitation rasters, or SnapWave mesh/output extras.


In [5]:
commands = [
    f"python -m sfincs_runs build_scenarios --config {config_yaml} --limit {scenario_limit} --include-waves --include-precip --zsini-mode boundary_t0 --force",
    "mkdir -p runs runs/debug",
    "sbatch --array=0-0 --export=ALL,EVENT_IDS=evt_0001,WORKERS=1,FORCE_RERUN=1,KEEP_STAGE=1,OMP_STACKSIZE=512M cluster/run_sfincs_dsai_wave_coupled.slurm",
    f"sbatch --array=0-{array_chunks - 1}%{max_concurrent_cpu_nodes} --export=ALL,WORKERS={workers_per_cpu_node},FORCE_RERUN=1,OMP_STACKSIZE=512M cluster/run_sfincs_dsai_wave_coupled.slurm",
]
for command in commands:
    print(command)

batch_plan = {
    "location": paths["location_name"],
    "config_yaml": str(config_yaml),
    "base_model_root": str(wave_base),
    "scenarios_root": str(paths["scenarios_root"]),
    "storage_root": str(paths["storage_root"]),
    "run_root": str(paths["run_root"]),
    "scenario_limit": scenario_limit,
    "workers_per_cpu_node": workers_per_cpu_node,
    "array_chunks": array_chunks,
    "max_concurrent_cpu_nodes": max_concurrent_cpu_nodes,
    "commands": commands,
    "slurm_scripts": [str(repo_root / "cluster" / "run_sfincs_dsai_wave_coupled.slurm")],
    "slurm_log_dir": str(repo_root / "runs"),
    "debug_stage_dir": str(repo_root / "runs" / "debug"),
}
paths["run_root"].mkdir(parents=True, exist_ok=True)
plan_path = paths["run_root"] / "create_scenarios_cluster_plan.json"
plan_path.write_text(json.dumps(batch_plan, indent=2), encoding="utf-8")
display(pd.Series({"plan": str(plan_path), "slurm_script_count": len(batch_plan["slurm_scripts"])}))


python -m sfincs_runs build_scenarios --config /home/grahamhults/projects/Flood-RM/locations/marshfield/config.yaml --limit 500 --include-waves --include-precip --zsini-mode boundary_t0 --force
mkdir -p runs runs/debug
sbatch --array=0-0 --export=ALL,EVENT_IDS=evt_0001,WORKERS=1,FORCE_RERUN=1,KEEP_STAGE=1,OMP_STACKSIZE=512M cluster/run_sfincs_dsai_wave_coupled.slurm
sbatch --array=0-15%4 --export=ALL,WORKERS=1,FORCE_RERUN=1,OMP_STACKSIZE=512M cluster/run_sfincs_dsai_wave_coupled.slurm


plan                  /home/grahamhults/projects/Flood-RM/locations/...
slurm_script_count                                                    1
dtype: object